# Port Tariff Calculator — exploration

This notebook is a **consumer** of the `tariffs` package. It defines no formulas, no rate values and no logic of its own — everything below just calls the engine and inspects its output. It is optional and not part of the graded path; the tests are (see `tests/`).

In [1]:
%load_ext autoreload
%autoreload 


In [2]:
from datetime import datetime

from tariffs.engine import calculate
from tariffs.models import VesselCall, Port, VesselType, PeriodBasis
from tariffs.adapter import to_assignment_output

## Reference case: SUDESTADA, Durban

Inputs as reconciled in `SPEC.md` §2 — GT 51,255 (not the vessel sheet's 51,300) and chargeable period 3.396 days (not the sheet's truncated 3.39).

In [4]:
call = VesselCall(
    vessel_name="SUDESTADA",
    port=Port.DURBAN,
    gross_tonnage=51255,
    length_overall_m=229.2,
    vessel_type=VesselType.BULK_CARRIER,
    arrival=datetime(2024, 11, 15, 10, 12),
    departure=datetime(2024, 11, 22, 13, 0),
    chargeable_period_days=3.396,
    chargeable_period_basis=PeriodBasis.DAYS_ALONGSIDE_PROXY,
    number_of_operations=2,
)

result = calculate(call)
result.totals()

{'light_dues': 60062.04,
 'port_dues': 199549.22,
 'towage_dues': 147074.38,
 'vts_dues': 33315.75,
 'pilotage_dues': 47189.94,
 'berthing_services': 19639.5,
 'running_of_vessel_lines': None}

Few additional tests created by claude in the main conversation:


Case 1 — Reference. Durban, GT 51,255, 2 ops, 3.396 days.
The known answers: 60,062.04 / 199,549.22 / 147,074.38 / 33,315.75 / 47,189.94 / 19,639.50. Run it with 51,300 and 3.39 too and confirm you get the documented deviations, not something else.

Case 2 — Cape Town, GT 8,500, 2 ops, 2.5 days.
Tests the low towage band and the 0.54 VTS rate. Cape Town has its own column in all four port-specific tables.

Light 9,951.80 · VTS 4,590.00 · Pilotage 14,418.78 · Towage 41,099.04 · Berthing 8,641.06 · Port dues 28,662.42 · Total 107,363.10

Case 3 — Saldanha, GT 150,000, 2 ops, 4 days.
Top towage band, above 100,000. Saldanha is the second port on the 0.65 VTS rate — a good check that it isn't hardcoded to Durban.

Light 175,620.00 · VTS 97,500.00 · Pilotage 60,327.14 · Towage 270,975.26 · Berthing 58,922.68 · Port dues 635,835.00 · Total 1,299,180.08

Case 4 — East London, GT 25,000, 2 ops, 3 days.
The column-alignment trap. East London has its own towage column but no pilotage column, so pilotage must fall to "Other" (6,547.45 + 10.49) and berthing to "Other Ports". If either picks up a named column instead, this case catches it.

Light 29,270.00 · VTS 13,500.00 · Pilotage 18,339.90 · Towage 75,914.82 · Berthing 12,443.82 · Port dues 91,525.00 · Total 240,993.54

Case 5 — Richards Bay, GT exactly 10,000, 2 ops, 1.5 days.
The band boundary. Under the spec's convention (lower exclusive, upper inclusive) this sits in the 2,001–10,000 band, giving towage 70,092.54. If the implementation puts it in the next band you get 79,999.76 — a 14% error that no other case reveals.

Light 11,708.00 · VTS 5,400.00 · Pilotage 64,106.92 · Towage 70,092.54 · Berthing 9,043.78 · Port dues 27,941.50 · Total 188,292.74

In [ ]:
port=Port.EAST_LONDON
gross_tonnage=25000
chargeable_period_days=3

In [28]:
call = VesselCall(
    vessel_name="SUDESTADA",
    port=port,
    gross_tonnage=gross_tonnage,
    length_overall_m=1111,
    vessel_type=VesselType.BULK_CARRIER,
    arrival=datetime(2024, 11, 21, 10, 12),
    departure=datetime(2024, 11, 22, 13, 0),
    chargeable_period_days=chargeable_period_days,
    chargeable_period_basis=PeriodBasis.DAYS_ALONGSIDE_PROXY,
    number_of_operations=2,
    # engaged_in_cargo_working=engaged_in_cargo_working,
    call_purpose_bunkers_stores_water_only=call_purpose_bunkers_stores_water_only,
)

result = calculate(call)
result.totals()

{'light_dues': 11708.0,
 'port_dues': 11176.6,
 'towage_dues': 68306.38,
 'vts_dues': 6500.0,
 'pilotage_dues': 39161.22,
 'berthing_services': 8339.82,
 'running_of_vessel_lines': None}

In [25]:
port=Port.DURBAN
gross_tonnage=10000
chargeable_period_days=1.5
# engaged_in_cargo_working=False
call_purpose_bunkers_stores_water_only=True

A — 35% port dues reduction. Durban, GT 20,000, 5 days, engaged_in_cargo_working=False.
Base 96,336.00 → 62,618.40. Only port dues changes; the other five stay at base.

B — 60% reduction. Durban, GT 10,000, 1.5 days (36h), call_purpose_bunkers_stores_water_only=True.
Base 27,941.50 → 11,176.60.

Then set engaged_in_cargo_working=False as well, so both the 35% and 60% conditions hold. The result must stay 11,176.60 — the 60% wins and the 35% is dropped, never compounded. If you get 40,742 (both applied) or 62,618 (35% won), the precedence is wrong.

C — 15% stacking. Same vessel, stay 0.4 days (9.6h), bunkers only.
Base 21,584.60, then 0.40 × 0.85 → 7,338.76. If it comes back as 21,584.60 × (1 − 0.60 − 0.15) = 5,396.15, the 15% was applied additively instead of multiplicatively.

D — 20% long-stay surcharge. Durban, GT 5,000, 40 days, not cargo working, no repairs.
Basic 9,636.50 (unsurcharged) + incremental 115,580.00 × 1.20 = 148,332.50. If you get 173,899, the surcharge hit the basic fee too.

Watch what happens with the 35% here. The book grants it for vessels not cargo working "for the first 30 days only", and at 40 days it's ambiguous whether the reduction applies to the first 30 days' portion, lapses entirely, or applies throughout. See what your implementation does and pick a reading — that's another README interpretation.

E — 25% towage out of hours. East London, GT 25,000, 2 ops, service on a Sunday.
37,957.41 × 1.25 = 47,446.76 per service → 94,893.53. Run the identical case at Durban and confirm no surcharge: 75,914.82. That's the 24-hour port check.

F — 50% pilotage out of hours. Same East London Sunday call.
9,169.95 × 1.5 = 13,754.93 per service. Note pilotage falls under "Other" at East London.

G — additional tug. East London GT 25,000, allocation is 2 craft, additional_tug_requested=True.
37,957.41 × 1.5 = 56,936.12 per service. Then set the vessel to GT 60,000 where the allocation is 3 and check the threshold moves.

## The full trace

One row per calculation step: the tariff, its section/page reference, the inputs used, the rounding applied, and the subtotal it produced. This is what lets a reviewer confirm the engine is doing what it claims, rather than just calling it.

In [4]:
result.trace_df()

,tariff,section,page,description,inputs,rounding,modifier,modifier_resolution,subtotal
0,Light dues,1.1.1,9,ceil(GT/100) x rate_per_100t (foreign/other ve...,"{'gross_tonnage': 51255.0, 'units': 513, 'rate...",ceil_per_100_t,None,None,60062.04
1,Port dues,4.1.1,21,basic = ceil(GT/100) x basic_rate_per_100t,"{'gross_tonnage': 51255.0, 'units': 513, 'basi...",pro_rata_time,None,None,98870.49
2,Port dues,4.1.1,21,incremental = ceil(GT/100) x incremental_rate_...,"{'chargeable_period_days': 3.396, 'chargeable_...",pro_rata_time,None,None,100678.73
3,Towage dues,3.6,15,"banded_base_plus_increment(GT, durban bands) x...","{'gross_tonnage': 51255.0, 'port': 'durban', '...",ceil_per_100_t,None,None,147074.38
4,VTS dues,2.1.1,11,GT x rate_per_gt (RoundingMode.EXACT — no ceil...,"{'gross_tonnage': 51255.0, 'port': 'durban', '...",exact,None,None,33315.75
5,Pilotage dues,3.3,13,(base_fee + ceil(GT/100) x per_100t) x 2 servi...,"{'gross_tonnage': 51255.0, 'port': 'durban', '...",ceil_per_100_t,None,None,47189.94
6,Berthing services (§3.8),3.8,18,(base_fee + ceil(GT/100) x per_100t) x 2 servi...,"{'gross_tonnage': 51255.0, 'port': 'durban', '...",ceil_per_100_t,None,None,19639.50
7,Running of vessel lines (§3.9),3.9,19,"Parsed, not calculated in v1 (SPEC.md §7.6).",{'mooring_boat_used': None},NaN,None,None,NaN


## Any warnings?

Empty here — the reference case doesn't set `mooring_boat_used`, so no §3.9 warning fires.

In [5]:
result.warnings()

[]

## Assignment-shaped output

The adapter renames domain results onto the assignment's output slots. Note `running_of_vessel_lines` is populated from berthing services (§3.8), with the §3.8/§3.9 mapping note carried in the output itself — and §3.9 is still surfaced separately, never suppressed.

In [6]:
to_assignment_output(result)

{'light_dues': {'amount': 60062.04, 'currency': 'ZAR', 'warnings': []},
 'port_dues': {'amount': 199549.22, 'currency': 'ZAR', 'warnings': []},
 'towage_dues': {'amount': 147074.38, 'currency': 'ZAR', 'warnings': []},
 'vts_dues': {'amount': 33315.75, 'currency': 'ZAR', 'warnings': []},
 'pilotage_dues': {'amount': 47189.94, 'currency': 'ZAR', 'warnings': []},
 'running_of_vessel_lines': {'amount': 19639.5,
  'currency': 'ZAR',
  'warnings': [],
  'note': 'The supplied benchmark value of ZAR 19,639.50 reconciles exactly to Tariff Book §3.8 Berthing Services (Other Ports), not to §3.9 Running of Vessel Lines, which would give ZAR 3,309.12 for two services. The benchmark figure is reported here; both sections are implemented separately in the domain model.'},
 'running_of_vessel_lines_section_3_9_not_calculated': {'amount': None,
  'currency': 'ZAR',
  'warnings': []}}

## A vessel with a mooring boat used

Setting `mooring_boat_used=True` doesn't change any calculated amount in v1 — it adds a warning instead (§3.9 is parsed, not calculated).

In [7]:
call_with_mooring_boat = call.model_copy(update={"mooring_boat_used": True})
result_with_mooring_boat = calculate(call_with_mooring_boat)
result_with_mooring_boat.warnings()

['A §3.9 Running of Vessel Lines charge applies (mooring_boat_used=True) but is not calculated in this version — see SPEC.md §7.6 / README.']